In [83]:
# import Pkg; Pkg.add(["DataStructures", "OnlineStats"])
using Agents
using Random
import StatsBase: middle
import StaticArraysCore: SVector, MVector
using FLoops
using DataStructures: OrderedDict
import LinearAlgebra: norm
using Makie, Makie.Observables
using OnlineStats
import DataStructures: DefaultDict
using ThreadPinning
using TimerOutputs

In [84]:
function Base.:-(a::Tuple{Float64, Float64}, b::Tuple{Float64, Float64})
    return (a[1] - b[1], a[2] - b[2])
end

In [85]:
last_model_step_time::UInt64 = time_ns()
avg_model_step_duration::Observable{Mean{Float64}} = Observable(Mean(weight=HarmonicWeight(30)))

Observable(Mean: n=0 | value=0.0)
    0 => (sc::Observables.SetindexCallback)(x) @ Observables ~/.julia/packages/Observables/YdEbO/src/Observables.jl:148


In [86]:
abstract type ParticleColor end
struct Red <: ParticleColor end
struct Green <: ParticleColor end
struct Yellow <: ParticleColor end
struct Cyan <: ParticleColor end
struct Orange <: ParticleColor end

In [87]:
@agent Par ContinuousAgent{2} begin
    color::ParticleColor
    # pos + vel are predifined in ContinuousAgent
end

In [88]:


function initialize_model()
    extent::NTuple{2, Float64} = 3 .* (500.0, 500.0);
    space2d = ContinuousSpace(extent;
                              periodic=false,
                              # spacing=20,
                              );
    model = ABM(Par, space2d;
                            properties=push!(Dict{Symbol, Float64}(
                                    lhs_rhs=>rand(range)
                                    for (lhs_rhs, range) in properties),
                                :time_scale => 1.0),
                            )

    for c in [Red(), Green(), Orange(), Cyan()]
        for _ in 1:750
            vel =  (0., 0.)
            add_agent!(model, vel, c)
        end
    end

    model
end


initialize_model (generic function with 1 method)

In [89]:


color_interact(lhs::ParticleColor,  rhs::ParticleColor, m::ABM) =
    abmproperties(m)[Symbol(string(color_sym(lhs))*'_'*string(color_sym(rhs)))]

color_sym(p::Par) = color_sym(p.color)
color_sym(::ParticleColor) = :black
color_sym(::Green) = :green
color_sym(::Red) = :red
color_sym(::Orange) = :orange
color_sym(::Cyan) = :cyan
color_sym(::Yellow) = :yellow

properties=OrderedDict(
    :red_red       => -1:0.1:1,
    :red_green     => -1:0.1:1,
    :red_orange    => -1:0.1:1,
    :red_cyan      => -1:0.1:1,
    :green_red     => -1:0.1:1,
    :green_green   => -1:0.1:1,
    :green_orange  => -1:0.1:1,
    :green_cyan    => -1:0.1:1,
    :orange_red    => -1:0.1:1,
    :orange_green  => -1:0.1:1,
    :orange_orange => -1:0.1:1,
    :orange_cyan   => -1:0.1:1,
    :cyan_red      => -1:0.1:1,
    :cyan_green    => -1:0.1:1,
    :cyan_orange   => -1:0.1:1,
    :cyan_cyan     => -1:0.1:1,
    :viscosity     =>  0:.01:1,
)

OrderedDict{Symbol, StepRangeLen{Float64, Base.TwicePrecision{Float64}, Base.TwicePrecision{Float64}, Int64}} with 17 entries:
  :red_red       => -1.0:0.1:1.0
  :red_green     => -1.0:0.1:1.0
  :red_orange    => -1.0:0.1:1.0
  :red_cyan      => -1.0:0.1:1.0
  :green_red     => -1.0:0.1:1.0
  :green_green   => -1.0:0.1:1.0
  :green_orange  => -1.0:0.1:1.0
  :green_cyan    => -1.0:0.1:1.0
  :orange_red    => -1.0:0.1:1.0
  :orange_green  => -1.0:0.1:1.0
  :orange_orange => -1.0:0.1:1.0
  :orange_cyan   => -1.0:0.1:1.0
  :cyan_red      => -1.0:0.1:1.0
  :cyan_green    => -1.0:0.1:1.0
  :cyan_orange   => -1.0:0.1:1.0
  :cyan_cyan     => -1.0:0.1:1.0
  :viscosity     => 0.0:0.01:1.0

In [90]:





function update_vel!(agent::Par, model::ABM; viscosity::Union{Nothing, Float64}=nothing)
    force = sum(
        let g = color_interact(agent.color, other.color, model),
            d = euclidean_distance(agent, other, model)
            (0 < d < 80 ? (g / d .* (agent.pos - other.pos)) : zero(SVector{2, Float64}))
        end
        for other in Agents.nearby_agents(agent, model, 80);
        init = zero(SVector{2, Float64}),
    )
    # push away from border
    force += 0.1*(max.(40 .- agent.pos, 0)
                 - max.(40 .- (spacesize(model) - agent.pos), 0))

    # combine past velocity and current force
    viscosity = (isnothing(viscosity) ? abmproperties(model)[:viscosity] : viscosity)
    agent.vel = agent.vel * (1-viscosity) + force
end

update_vel! (generic function with 1 method)

In [91]:
agent_step!(agent, model) = move_agent!(agent, model, abmproperties(model)[:time_scale])

agent_step! (generic function with 1 method)

In [92]:



function model_step!(model)
    @timeit TimerOutput() "update_vel! loop" begin
        # about 20% speedup to extract the viscosity var first.
        viscosity::Float64 = abmproperties(model)[:viscosity]
        @floop for agent in collect(allagents(model))
            update_vel!(agent, model; viscosity=viscosity)
        end
    end
    @timeit TimerOutput() "max reduction" begin
        max_vel = maximum(map(x->norm(x.vel), allagents(model)))
    end
    @timeit TimerOutput() "mean reduction" begin
        mean_vel = mean(map(x->norm(x.vel), allagents(model)))
    end
    # model.time_scale = max(0.1/max_vel, 1.)
    if max_vel*model.time_scale > getfield(model, :space).spacing || mean_vel*model.time_scale > 30
        model.time_scale /= 1.1
    end
    if model.time_scale < 0.9
        model.time_scale *= 1.01
    elseif model.time_scale > 1.1
        model.time_scale /= 1.01
    end
    @debug model.time_scale

    global last_model_step_time
    global avg_model_step_duration
    delta_time = time_ns() - last_model_step_time
    last_model_step_time = time_ns()
    fit!(avg_model_step_duration[], delta_time/1e9)
    notify(avg_model_step_duration)  # to update the fps label
end

model_step! (generic function with 1 method)

In [93]:
function run_sim(; to=TimerOutput())
    # this is basically it, but we want to rearrange the layout a bit.
    model = make_model(to)
    fig, ax, abmobs = with_theme(theme_dark()) do
        abmplot(model;
                ac=color_sym, as=8.0,  # agent color and size
                params=ParticleLife.properties,
                scatterkwargs=(; :markerspace=>:data),
                enable_inspection=false)
    end

    # the rest is mostly changing the layout a bit and adding a randomize button and an fps label
    controls = content(fig[2,1][1,1])
    param_sliders = content(fig[2,1][1,2])
    update_button = content(param_sliders[2,1])

    # the size is currently fixed, probably not a good long-term solution
    ax.width[] = 1000
    ax.height[] = 1500
    ui = fig[1,2] = GridLayout();
    param_sliders.width[] = 500
    ui[1,1] = param_sliders
    sg::SliderGrid = content(param_sliders[1,1])
    update_and_rand_button = param_sliders[2,1] = GridLayout();
    update_and_rand_button[1,1] = update_button
    rand_button = Button(update_and_rand_button[1,2], label="randomize", tellwidth=true)
    on(rand_button.clicks) do _  # randomize params
        for s in sg.sliders
            set_close_to!(s, rand(s.range[]))
        end
        update_button.clicks[] += 1
        abmproperties(model)[:time_scale] = 1.0
    end
    Label(ui[2,1], "----------------------")
    ui[3,1] = controls
    Label(ui[4,1], "----------------------")
    # show fps
    empty!(avg_model_step_duration.listeners)
    fps = throttle(0.5, @lift 1/value($avg_model_step_duration))
    fps_label = Label(ui[5,1], text="0.0 fps")
    on(fps) do fps
        fps_label.text = "$(round(fps, digits=2)) fps"
    end

    Makie.deleterow!(fig.layout, 2)
    fig
end

run_sim (generic function with 1 method)

In [94]:
function custom_model_step!(model)
    scheduler = Schedulers
    for id in scheduler(model)
        agent_step!(model[id], model)
    end
    return model
end


custom_model_step! (generic function with 1 method)

In [98]:
using GLMakie
function make_video()
    with_theme(theme_dark()) do
        abmvideo("/tmp/foo.mp4", initialize_model(), ac=color_sym, as=8.0;
                scatterkwargs=(; :markerspace=>:data))
    end
end

make_video (generic function with 1 method)

In [99]:
make_video()

MethodError: MethodError: no method matching abmvideo(::String, ::StandardABM{ContinuousSpace{2, false, Float64, typeof(Agents.no_vel_update)}, Par, typeof(Agents.Schedulers.fastest), Dict{Symbol, Float64}, TaskLocalRNG}; ac::typeof(color_sym), as::Float64, scatterkwargs::@NamedTuple{markerspace::Symbol})
The function `abmvideo` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  abmvideo(::Any, ::Any, !Matched::Any; ...)
   @ AgentsVisualizations ~/.julia/packages/Agents/xtlGn/ext/AgentsVisualizations/src/convenience.jl:79
  abmvideo(::Any, ::Any, !Matched::Any, !Matched::Any; spf, framerate, frames, title, showstep, figure, axis, recordkwargs, kwargs...)
   @ AgentsVisualizations ~/.julia/packages/Agents/xtlGn/ext/AgentsVisualizations/src/convenience.jl:79
